# Lecture 8: Matrix Multiplications Kernels (Part A)

In [ ]:
!pip install --upgrade numba numba-cuda

## cuBLAS Kernel - Baseline

In [ ]:
%%writefile matmul-cublas.py
from numba import cuda, float32
import cupy as cp
import numpy as np

m = 1000
k = 1200
n = 800

matA = cp.ones(shape=(m, k), dtype=cp.float32) * 2.0
matB = cp.ones(shape=(k, n), dtype=cp.float32) * 3.0
matC = cp.zeros(shape=(m, n), dtype=cp.float32)

matC = cp.matmul(matA, matB)

matA = cp.asnumpy(matA)
matB = cp.asnumpy(matB)
matC = cp.asnumpy(matC)

np.testing.assert_allclose(matC, matA @ matB, rtol=1e-5, atol=1e-5)

In [ ]:
!ncu --set full -f -o matmul-cublas.ncu-rep python3 matmul-cublas.py

## Kernel 1 - Uncoalesced Global Stores


In [ ]:
%%writefile matmul-01.py
"""
load from matA -> coalesced
load from matB -> uncoalesced
store in matC -> uncoalesced
"""
from numba import cuda, float32
import cupy as cp
import numpy as np

ceil_div = lambda a, b: (a + b - 1) // b

@cuda.jit
def matmul(matA, matB, matC, m, n, k):
  out_row = cuda.threadIdx.x + cuda.blockDim.x * cuda.blockIdx.x
  out_col = cuda.threadIdx.y + cuda.blockDim.y * cuda.blockIdx.y
  if out_col >= n or out_row >= m:
    return
  dot_product = float32(0.0)
  for i in range(k):
    dot_product += matA[out_row, i] * matB[i, out_col]
  matC[out_row, out_col] = dot_product

m = 1000
k = 1200
n = 800

matA = cp.ones(shape=(m, k), dtype=cp.float32) * 2.0
matB = cp.ones(shape=(k, n), dtype=cp.float32) * 3.0
matC = cp.zeros(shape=(m, n), dtype=cp.float32)

threads_per_block = (32, 32)
blocks_per_grid = (
    ceil_div(m, threads_per_block[0]),
    ceil_div(n, threads_per_block[1])
)
matmul[blocks_per_grid, threads_per_block](matA, matB, matC, m, n, k)
cuda.synchronize()

matA = cp.asnumpy(matA)
matB = cp.asnumpy(matB)
matC = cp.asnumpy(matC)

np.testing.assert_allclose(matC, matA @ matB, rtol=1e-5, atol=1e-5)

In [ ]:
!ncu --set full -f -o matmul-01.ncu-rep python3 matmul-01.py

## Kernel 2 - Coalesced Global Stores

In [ ]:
%%writefile matmul-02.py
"""
load from matA -> coalesced
load from matB -> uncoalesced
store in matC -> coalesced
"""
from numba import cuda, float32
import cupy as cp
import numpy as np

ceil_div = lambda a, b: (a + b - 1) // b

@cuda.jit
def matmul(matA, matB, matC, m, n, k):
  out_col = cuda.threadIdx.x + cuda.blockDim.x * cuda.blockIdx.x
  out_row = cuda.threadIdx.y + cuda.blockDim.y * cuda.blockIdx.y
  if out_col >= n or out_row >= m:
    return
  dot_product = float32(0.0)
  for i in range(k):
    dot_product += matA[out_row, i] * matB[i, out_col]
  matC[out_row, out_col] = dot_product

m = 1000
k = 1200
n = 800

matA = cp.ones(shape=(m, k), dtype=cp.float32) * 2.0
matB = cp.ones(shape=(k, n), dtype=cp.float32) * 3.0
matC = cp.zeros(shape=(m, n), dtype=cp.float32)

threads_per_block = (32, 32)
blocks_per_grid = (
    ceil_div(n, threads_per_block[0]), # x-direction
    ceil_div(m, threads_per_block[1])  # y-direction
)
matmul[blocks_per_grid, threads_per_block](matA, matB, matC, m, n, k)
cuda.synchronize()

matA = cp.asnumpy(matA)
matB = cp.asnumpy(matB)
matC = cp.asnumpy(matC)

np.testing.assert_allclose(matC, matA @ matB, rtol=1e-5, atol=1e-5)

In [ ]:
!ncu --set full -f -o matmul-02.ncu-rep python3 matmul-02.py

## Kernel 3 - Tiled MatMul with Shared Memory

In [ ]:
%%writefile matmul-03.py
from numba import cuda, float32
import cupy as cp
import numpy as np

ceil_div = lambda a, b: (a + b - 1) // b
tile_dim = 32 # size of the tile

@cuda.jit
def matmul(matA, matB, matC, m, n, k):
  tileA = cuda.shared.array(shape=(tile_dim, tile_dim), dtype=cp.float32)
  tileB = cuda.shared.array(shape=(tile_dim, tile_dim), dtype=cp.float32)

  tx = cuda.threadIdx.x
  ty = cuda.threadIdx.y

  out_col = tx + cuda.blockIdx.x * cuda.blockDim.x
  out_row = ty + cuda.blockIdx.y * cuda.blockDim.y

  num_tiles = (k + tile_dim - 1) // tile_dim
  product = float32(0.0)
  for t in range(num_tiles):
    # phase 1:
    # threads within the block
    # will read elements from matA and matB (gmem)
    # and store them in tileA and tileB (smem)
    tile_offset = t * tile_dim
    b_row = tile_offset + ty
    a_col = tile_offset + tx

    # load values from matA (gmem) into tileA (smem)
    if out_row < m and a_col < k:
      tileA[ty, tx] = matA[out_row, a_col]
    else:
      tileA[ty, tx] = float32(0.0)

    if b_row < k and out_col < n:
      tileB[ty, tx] = matB[b_row, out_col]
    else:
      tileB[ty, tx] = float32(0.0)

    cuda.syncthreads()

    # phase 2:
    # threads will compute dot product
    # by reading elements from tileA and tileB (smem)
    for i in range(tile_dim):
      product += tileA[ty, i] * tileB[i, tx]

    cuda.syncthreads()

  if out_row < m and out_col < n:
    matC[out_row, out_col] = product

m = 1000
k = 1200
n = 800

matA = cp.ones(shape=(m, k), dtype=cp.float32) * 2.0
matB = cp.ones(shape=(k, n), dtype=cp.float32) * 3.0
matC = cp.zeros(shape=(m, n), dtype=cp.float32)

threads_per_block = (tile_dim, tile_dim)
blocks_per_grid = (
    ceil_div(n, threads_per_block[0]),
    ceil_div(m, threads_per_block[1])
)
matmul[blocks_per_grid, threads_per_block](matA, matB, matC, m, n, k)
cuda.synchronize()

matA = cp.asnumpy(matA)
matB = cp.asnumpy(matB)
matC = cp.asnumpy(matC)

np.testing.assert_allclose(matC, matA @ matB, rtol=1e-5, atol=1e-5)

In [ ]:
!ncu --set full -f -o matmul-03.ncu-rep python3 matmul-03.py

## Kernel 4 - Wider Loads with Shared Memory Tiling

In [3]:
#%%writefile matmul-04.py
from numba import cuda, float32
import cupy as cp
import numpy as np

ceil_div = lambda a, b: (a + b - 1) // b
BM = 32
BN = 32
BK = 32
TM = 8                # no. of output elements processed by a single thread
NT = (BM * BN) // TM  # no. of threads required within a block

@cuda.jit
def load_tileA(tileA, thread_idx, block_row, bk_offset):
    # phase 1: load values from matA into tileA
    for load_idx_a in range(thread_idx, BM * BK, NT):
      # convert 'linear' tid
      # to 2D coordinates (r, c)
      tile_a_row = load_idx_a // BK
      tile_a_col = load_idx_a % BK
      matA_row = block_row + tile_a_row
      matA_col = bk_offset + tile_a_col
      if matA_col < k and matA_row < m:
        tileA[tile_a_row, tile_a_col] = matA[matA_row, matA_col]
      else:
        tileA[tile_a_row, tile_a_col] = float32(0.0)

@cuda.jit
def load_tileB(tileB, thread_idx, block_col, bk_offset):
    # phase 1: load values from matB into tileB
    for load_idx_b in range(thread_idx, BK * BN, NT):
      tile_b_row = load_idx_b // BN
      tile_b_col = load_idx_b % BN
      matB_row = bk_offset + tile_b_row
      matB_col = block_col + tile_b_col
      if matB_row < k and matB_col < n:
        tileB[tile_b_row, tile_b_col] = matB[matB_row, matB_col]
      else:
        tileB[tile_b_row, tile_b_col] = float32(0.0)

@cuda.jit
def compute_dot_products(tileA, tileB, thread_col, thread_row_base, thread_results):
    for dot_k in range(BK):
        b_val = tileB[dot_k, thread_col]
        for tm_offset in range(TM):
            thread_results[tm_offset] += tileA[thread_row_base + tm_offset, dot_k] * b_val

@cuda.jit
def write_thread_results(thread_results, matC, block_row, block_col, thread_col, thread_row_base):
  out_col = block_col + thread_col
  for tm_offset in range(TM):
    out_row = block_row + thread_row_base + tm_offset
    if out_row < m and out_col < n:
      matC[out_row, out_col] = thread_results[tm_offset]

@cuda.jit
def matmul(matA, matB, matC, m, n, k):
  tileA = cuda.shared.array(shape=(BM, BK), dtype=cp.float32)
  tileB = cuda.shared.array(shape=(BK, BN), dtype=cp.float32)

  tid = cuda.threadIdx.x
  thread_col = tid % BN
  thread_row_base = (tid // BN) * TM

  block_row = cuda.blockIdx.y * BM
  block_col = cuda.blockIdx.x * BN

  # register allocation and initialization
  thread_results = cuda.local.array(TM, dtype=cp.float32)
  for tm_offset in range(TM):
    thread_results[tm_offset] = float32(0.0)


  for bk_offset in range(0, k, BK):
    # phase 1:
    # threads will load values from matA and matB
    # into tileA and tileB
    # gmem -> smem
    load_tileA(tileA, tid, block_row, bk_offset)
    load_tileB(tileB, tid, block_col, bk_offset)

    cuda.syncthreads()

    # phase 2
    compute_dot_products(tileA, tileB, thread_col, thread_row_base, thread_results)

    cuda.syncthreads()

  write_thread_results(thread_results, matC, block_row, block_col, thread_col, thread_row_base)

m = 1000
k = 1200
n = 800

matA = cp.ones(shape=(m, k), dtype=cp.float32) * 2.0
matB = cp.ones(shape=(k, n), dtype=cp.float32) * 3.0
matC = cp.zeros(shape=(m, n), dtype=cp.float32)

threads_per_block = (
    NT, 1
)
blocks_per_grid = (
    ceil_div(n, BN),
    ceil_div(m, BM)
)
matmul[blocks_per_grid, threads_per_block](matA, matB, matC, m, n, k)
cuda.synchronize()

matA = cp.asnumpy(matA)
matB = cp.asnumpy(matB)
matC = cp.asnumpy(matC)

np.testing.assert_allclose(matC, matA @ matB, rtol=1e-5, atol=1e-5)

In [ ]:
!ncu --set full -f -o matmul-04.ncu-rep python3 matmul-04.py